***

* [总目录](../0_Introduction/0_introduction.ipynb)
* [术语表](../0_Introduction/1_glossary.ipynb)
* [9. 实践部分](9_1_visualisation-inspection.ipynb)
    * 上一节： [9.13 高级谱线分析](9_13_advanced_spectral_line_analysis.ipynb)
    * 下一节： [9.x 延伸阅读与后续实践方向](9_x_further_reading_and_workflow.ipynb)

***


## 9.14 端到端连续谱教学案例：从数据检查到科学通量

前面各节分别讨论了数据检查、校准、成像、自校准、图像质量和测量。本节把这些部件重新串成一个端到端连续谱案例。案例的科学目标很具体：在一个含有明亮紧致源和弱扩展源的目标场中，可靠测量弱扩展源的积分通量，并判断这个通量是否已经达到可以用于科学解释的质量。

这个目标看似简单，却几乎覆盖了连续谱处理的核心判断。明亮紧致源会放大旁瓣和校准误差，弱扩展源对权重、CLEAN mask、残差背景和通量恢复很敏感；若早期 flagging 或校准不稳，最终通量可能看似有数值，实则没有可靠误差边界。因此，本案例的重点不是复述软件命令，而是训练一条判断链：每一步产生什么中间产物，如何判断产物是否可信，错误会怎样传播到最终科学量。


### 9.14.1 案例结构：每一步都必须留下诊断产物

端到端处理不应被理解成“按顺序运行一串任务”。更稳健的理解是：每一步都把数据状态、模型状态或解释边界向前推进一点，并留下可检查的诊断产物。若某一步没有诊断产物，它就很难被教学和复现；若只有产物没有判断标准，它就仍然停留在演示层面。

下图把本案例拆成六个环节。第一个环节检查 Measurement Set 的结构和 $uv$ 覆盖；第二个环节识别 RFI、坏天线或异常扫描；第三个环节建立带通、时间增益和通量标尺；第四个环节从 dirty image 进入 CLEAN/restored image；第五个环节在模型足够可靠时做保守自校准；第六个环节用 beam-aware 的方式测量弱扩展源通量，并建立最小误差预算。


![端到端连续谱案例的处理链](figures/practical_end2end_workflow_map.png)

**图 9.14.1** 端到端连续谱案例的处理链。每个环节都对应一个诊断产物和一个判断问题：数据是否可用、flag 是否合理、校准解是否稳定、图像是否受旁瓣或残差限制、自校准是否已经越过安全边界、最终测量是否有可信误差预算。


### 9.14.2 数据检查与 flagging：先判断数据能不能被求解

处理链的起点是数据检查，而不是校准任务。Measurement Set 的核心可见度列、权重、flag、天线表、谱窗表和扫描表共同决定后续求解的可行性。一个常见误区是直接进入 bandpass 或 gaincal，然后再用校准失败来倒推数据质量问题。更可靠的做法是在求解前先回答三个问题：哪些时间和频率范围明显不应参与求解；哪些异常集中在单天线或少数基线上；哪些结构可能是天体信号而不是坏数据。

本案例假设数据中存在两类典型问题：一段窄频 RFI 和一台天线的短时异常。窄频 RFI 会在按通道统计的 flag fraction 中形成局部峰值，也会污染带通解；单天线异常则常表现为若干条共享同一根天线的基线同时变坏，并在按时间统计的 flag fraction 中形成短时峰值。若这些异常进入校准求解，解会试图把坏数据吸收到天线增益中，随后再把错误解转移到目标场。


![端到端案例中的 flagging 与校准诊断](figures/practical_end2end_flag_calibration_diagnostics.png)

**图 9.14.2** flagging 与基础校准诊断。上排显示按通道和按时间统计的 flag fraction；下排显示带通振幅和相位增益诊断。RFI 通道和坏时间段不应只被视为“图上不好看”，而应进入校准解可信度判断。


从教学角度看，这里最实用的训练问题不是“flag 多少比例才算合适”，而是“flag 后是否保留了足够的校准约束”。过少 flag 会污染解，过度 flag 会破坏 $uv$ 覆盖、降低灵敏度，并可能让某些天线或时间段缺乏可解性。较稳健的判断应同时看三类量：剩余数据的时间频率覆盖、每根天线的有效数据量，以及校准解是否变得更平滑、更可解释。

对真实 CASA 工作流而言，这一段通常对应 `listobs`、`plotants`、`plotms`、`flagdata` 和校准前后的 `plotcal` 检查；对 WSClean/DP3 或其他工具链而言，名称可能不同，但判断对象相同：哪些样本参与求解，哪些样本参与成像，权重和 flag 是否仍然反映真实数据质量。


### 9.14.3 校准链：把误差模型拆成可解的部分

基础连续谱校准通常把观测可见度写成

$$
V_{pq}^{\mathrm{obs}}(t,\nu)=G_p(t)B_p(\nu)X_{pq}(t,\nu)B_q^*(\nu)G_q^*(t)+N_{pq},
$$

其中 $B_p(\nu)$ 描述天线 $p$ 的频率响应，$G_p(t)$ 描述时间变化的复增益，$X_{pq}$ 是由天空模型预测的理想相干量，$N_{pq}$ 是噪声和未建模残差。这个写法的价值在于把一个看似复杂的问题拆成几个物理变化轴：带通主要随频率变化，时间增益主要随时间变化，通量标尺负责把相对振幅放到绝对单位上。

真实处理时，校准顺序不是任意的。带通源需要足够亮且谱形平滑，否则逐通道解会被噪声或 RFI 支配；相位校准源需要靠近目标场并反复观测，否则时间插值会失去物理意义；通量校准源需要外部模型支持，否则最终 Jy 标尺会漂移。若这些条件不满足，校准仍可能给出一组数值解，但这些解不一定具有可转移性。

本案例强调一个实用判断：校准解应当比原始数据更接近“物理上可解释的低维变化”。带通振幅应当在无 RFI 的通道上平滑变化，相位增益应当在解间隔之间连续，通量标尺不应由目标场自校准随意改变。若解出现尖峰、跳变、单天线离群或与观测结构无关的快速振荡，问题往往不在绘图，而在数据选择、模型、参考天线或求解间隔。


### 9.14.4 成像与自校准：图像变好不等于科学量已经可信

完成基础校准后，成像阶段把采样可见度转换为图像域产品。dirty image 可以近似写成

$$
I^{\mathrm{dirty}}(l,m)=I(l,m)*B^{\mathrm{dirty}}(l,m)+N(l,m),
$$

其中 $B^{\mathrm{dirty}}$ 是由采样函数和权重决定的 dirty beam。这个公式说明，dirty image 中亮源周围的条纹和旁瓣不应直接解释为天空结构。CLEAN 的作用是寻找一个模型，使模型与 dirty beam 卷积后能够解释 dirty image 的主要结构；restored image 则把 CLEAN 分量与恢复波束卷积后再加回残差。

自校准进一步使用目标场模型反求天线增益。它能显著改善动态范围，但也引入一个危险：若模型不完整，求解器可能把真实天空结构或未清理旁瓣吸收到增益中。因而自校准的起点通常应是保守的 phase-only 解；只有当相位自校准后模型稳定、残差接近噪声、外部通量标尺可信时，才考虑幅相自校准。图像变得更平滑或峰值更高，并不自动等价于科学量更可信。


![端到端案例中的成像与自校准前后对比](figures/practical_end2end_imaging_selfcal_comparison.png)

**图 9.14.3** 从输入天空、dirty image、restored image 到自校准后残差的对比。明亮紧致源会把旁瓣和相位误差放大到全场；弱扩展源的测量必须在旁瓣、残差和噪声都受控之后才有意义。


这张图适合训练三个层次的判断。第一层是图像域直觉：dirty image 中的条纹主要来自采样函数和相位误差，而不是真实天空。第二层是参数判断：weighting、CLEAN mask、threshold、niter 和 self-cal solution interval 都会改变残差形态。第三层是科学判断：即使 self-cal 后图像外观改善，弱扩展源的总通量仍可能受缺短间距、残差背景、主波束模型和 CLEAN mask 影响。


### 9.14.5 从图像到科学量：通量测量必须带着误差预算

射电连续谱图像常以 Jy/beam 为单位。若对扩展源做孔径积分，不能简单把像素值相加当作 Jy，而应除以合成波束覆盖的像素数：

$$
S_{\mathrm{ap}} \simeq \frac{\sum_{i\in A} I_i}{\Omega_{\mathrm{beam}}/\Omega_{\mathrm{pix}}},
$$

其中 $A$ 是孔径区域，$\Omega_{\mathrm{beam}}$ 是合成波束面积，$\Omega_{\mathrm{pix}}$ 是单个像素的角面积。若源位于主波束边缘，还需要进行 primary beam correction，并把主波束模型误差加入系统误差。

一个最小但实用的误差预算可写成

$$
\sigma_S^2 \simeq \sigma_{\mathrm{thermal}}^2+\sigma_{\mathrm{scale}}^2+
\sigma_{\mathrm{PB}}^2+\sigma_{\mathrm{deconv}}^2+\sigma_{\mathrm{bg}}^2.
$$

这些项分别对应热噪声、通量标尺、主波束模型、去卷积/残差和背景估计。真实工作中各项未必完全独立，但这个表达式有教学价值：它强迫分析者说明最终误差从哪里来，而不是只给出一个没有来源的通量数字。


![端到端案例中的通量恢复与误差预算](figures/practical_end2end_science_measurement_uncertainty.png)

**图 9.14.4** 弱扩展源的孔径测量、通量恢复和最小误差预算。dirty image 的孔径通量明显偏低；CLEAN 和保守自校准后通量接近真值，但最终可信度仍由热噪声、通量标尺、主波束模型、去卷积残差和背景估计共同限制。


本案例给出的数值不是某个望远镜的标准答案，而是一个用于训练判断的量级示例。dirty aperture 只恢复约 $70\%$ 的通量，说明旁瓣和残差背景会系统性影响弱扩展源；CLEAN 后通量接近 $95\%$，保守 self-cal 后接近真值；总误差预算约为 $8$--$9\%$。若科学问题要求 $5\%$ 以内的通量精度，那么这个结果还不能直接作为最终结论；若科学问题只需判断扩展源是否存在或通量量级是否合理，则可能已经足够。


### 9.14.6 分层教学目标：同一个案例可以读出不同深度

这个端到端案例可以服务不同层次的训练。基础层次的重点是建立物理图像：坏数据会污染校准，校准误差会进入图像，dirty beam 会制造旁瓣，Jy/beam 图像不能直接逐像素求和。中级层次的重点是参数选择和诊断：flagging 阈值、解间隔、参考天线、CLEAN mask、weighting 和 self-cal 停止条件都会改变最终图像。研究训练层次的重点则是系统误差和科学边界：通量标尺、主波束、模型不完备、缺短间距和 correlated noise 如何进入最终结论。

因此，案例的教学使用不应只有“完成流程”这一种目标。更有价值的使用方式是比较不同选择的后果。例如保留 RFI 通道会怎样改变带通解；过度 flag 会怎样破坏 $uv$ 覆盖；过深 CLEAN 会怎样把噪声吸入模型；过早 amplitude self-cal 会怎样改变通量标尺；在主波束边缘测量弱源时，系统误差怎样超过热噪声。


### 9.14.7 与真实软件工作流的对应

在 CASA 中，这个案例可对应到一条典型任务链：`listobs` 和 `plotms` 用于检查观测结构和异常数据；`flagdata` 用于形成可追踪的 flag 版本；`setjy`、`bandpass`、`gaincal`、`fluxscale`、`applycal` 用于基础校准；`tclean` 用于成像和去卷积；`gaincal` 与 `applycal` 再次进入保守自校准；`imstat`、`imfit`、`imview` 或 CARTA 用于图像 QA 和测量。WSClean、DP3、CARTA 等工具链的任务名称不同，但对应的物理对象仍是数据、flag、校准解、模型、权重、图像和测量。

教材案例不应把这些任务写成唯一流程。真实数据处理需要保留处理日志、参数文件、关键诊断图和失败版本。可复现实操材料的核心不是“最终图像很好看”，而是任何一步出现问题时，能够回到相应诊断产物，判断问题来自数据质量、校准模型、成像参数还是科学测量假设。


### 9.14.8 本案例的结论

端到端连续谱处理的核心能力，是把每一步的技术操作翻译成物理和统计判断。数据检查决定哪些样本可以信任，校准决定误差模型是否可转移，成像决定不完整 Fourier 采样如何进入图像，自校准决定模型和增益是否相互污染，最终测量决定图像产品是否足以支撑科学结论。

第 9 章前面的各节分别训练这些能力；本节把它们放回一个完整案例中。后续继续扩展实践材料时，也应尽量采用这种结构：从具体科学问题出发，用可复现数据和静态诊断图组织流程，并且始终给出“如何知道做对了”的判断标准。

***
